In [3]:
import numpy as np
import pandas as pd
from scipy import stats

In [6]:
active_validators_size = pd.read_csv('../int/active_validators_size_change.csv')
active_validators_category = pd.read_csv('../int/active_validators_category_change.csv')

active_validators_category = active_validators_category.drop(columns=('Unnamed: 0'))
active_validators_size = active_validators_size.drop(columns=('Unnamed: 0'))

In [8]:
active_validators_category

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,300.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
1,600.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
2,900.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
3,1200.0,0.000000,NaN,0.000000,0.043309,0.000000,0.049628,0.037981
4,1500.0,0.000000,NaN,0.000000,0.000000,0.000000,0.283447,0.189834
...,...,...,...,...,...,...,...,...
29948,8984700.0,-0.004689,0.102837,-0.001206,0.000000,-0.003543,-0.025814,-0.000599
29949,8985000.0,0.000000,0.065491,-0.000603,0.000000,-0.015942,0.007594,0.005988
29950,8985300.0,0.000000,0.000000,-0.001206,0.000000,0.000000,0.026578,0.006586
29951,8985600.0,0.000391,0.019250,0.000000,0.000000,0.008858,0.022016,0.007883


In [7]:
active_validators_size = active_validators_size.drop(columns=(['100+', 'total']))

,slot,1,100+,2-5,20-99,6-19,total
0,300.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1,600.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2,900.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
3,1200.0,0.294118,0.000000,0.095329,0.0,0.218938,0.037981
4,1500.0,0.000000,0.286786,0.000000,0.0,0.054615,0.189834
...,...,...,...,...,...,...,...
29948,8984700.0,-0.030469,-0.000326,0.000000,0.0,0.000000,-0.000599
29949,8985000.0,0.020319,0.006306,0.000000,0.0,0.000000,0.005988
29950,8985300.0,0.000000,0.007175,0.000000,0.0,0.000000,0.006586
29951,8985600.0,0.000000,0.008370,0.010076,0.0,0.005530,0.007883


In [11]:
rewards_size = pd.read_csv('../int/rewards_size.csv', usecols=('1', '2-5', '6-19', '20-99', '100+', 'slot'))
rewards_category = pd.read_csv('../int/rewards_category.csv', usecols=('slot', 'CEX', 'Liquid Restaking', 'Liquid Staking', 'Solo Stakers', 'Staking Pools'))

columns_to_compare = ['1', '2-5', '20-99', '6-19']
reference_column = '100+'

# Calculate the percentage difference compared to '100+'
for col in columns_to_compare:
    rewards_size[f'{col}'] = ((rewards_size[col] - rewards_size[reference_column]) / rewards_size[reference_column])

# Display the updated DataFrame
rewards_size = rewards_size.drop(columns=('100+'))


# List of columns to calculate the percentage difference for
columns_to_compare = ['CEX', 'Liquid Restaking', 'Liquid Staking', 'Staking Pools']
reference_column = 'Solo Stakers'

# Calculate the percentage difference compared to 'Solo Stakers'
for col in columns_to_compare:
    rewards_category[f'{col}'] = ((rewards_category[col] - rewards_category[reference_column]) / rewards_category[reference_column])

# Display the updated DataFrame
rewards_category = rewards_category.drop(columns=('Solo Stakers'))

In [15]:
active_validators_size = active_validators_size.drop(columns=(['100+', 'total']))

In [16]:
import numpy as np
import pandas as pd
from scipy import stats

# Assuming the DataFrames are already defined and loaded
# active_validators_size and rewards_size

# Calculate the percent change for the rewards_size DataFrame
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the DataFrames on the slot column for percent change calculations
price_elasticity_size = pd.merge(active_validators_size, rewards_size_pct_change, on='slot')

# Replace infinite values with NaN and drop rows with NaN values for elasticity calculations
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)
price_elasticity_size.dropna(inplace=True)

# Merge the original DataFrames on the slot column for correlation calculations
merged_df = pd.merge(active_validators_size, rewards_size, on='slot', suffixes=('_exit', '_apy'))

# Replace infinite values with NaN for correlation calculations
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Initialize dictionaries to store the results
elasticity = {}
t_stats = {}
standard_deviations = {}
p_values = {}
counts = {}
correlations = {}

# List of columns to calculate elasticity and correlation for
columns = active_validators_size.columns.drop(['slot'])
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'{col}_x'] / price_elasticity_size[f'{col}_y']
    
    # Drop NaN values for the specific column for elasticity calculations
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    # Store the results for elasticity
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    counts[col] = valid_elasticity.count()
    
    # Perform a one-sample t-test against zero
    t_stat, p_value = stats.ttest_1samp(valid_elasticity, 0)
    t_stats[col] = t_stat
    p_values[col] = p_value
    
    # Calculate the correlation coefficient using non-percent change values
    correlation, _ = stats.pearsonr(merged_df[f'{col}_exit'], merged_df[f'{col}_apy'])
    correlations[col] = correlation

# Print the results
print("Elasticity and Correlation Analysis Results:")
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t-stat = {t_stats[col]}, Std Dev = {standard_deviations[col]}, P-value = {p_values[col]}, Count = {counts[col]}, Correlation Coefficient = {correlations[col]}')

# Display the DataFrame with elasticity columns (optional)
print(price_elasticity_size)

Elasticity and Correlation Analysis Results:
1: Mean Elasticity = 0.004038318181545583, t-stat = 0.1619324796103517, Std Dev = 0.4305015309246463, P-value = 0.8714691330867995, Count = 298, Correlation Coefficient = 0.061259042063267606
2-5: Mean Elasticity = 0.0013565355242780217, t-stat = 0.15010253044299426, Std Dev = 0.15600958790949088, P-value = 0.8807856083196195, Count = 298, Correlation Coefficient = 0.056929493290690004
20-99: Mean Elasticity = 0.06156446733105811, t-stat = 1.8945052290168412, Std Dev = 0.5609736343049726, P-value = 0.05912924623560406, Count = 298, Correlation Coefficient = -0.0898549411853028
6-19: Mean Elasticity = -0.0067631639932466455, t-stat = -0.5780053260797242, Std Dev = 0.20198829816111352, P-value = 0.5636984984303959, Count = 298, Correlation Coefficient = -0.04168159482360834
          slot       1_x     2-5_x   20-99_x    6-19_x       1_y     2-5_y  \
1    6847200.0  0.000000  0.000000  0.009888  0.000000  0.106203 -0.026545   
2    6854400.0  

In [18]:
import numpy as np
import pandas as pd
from scipy import stats

# Assuming the DataFrames are already defined and loaded
# active_validators_category and rewards_category

# Calculate the percent change for the rewards_category DataFrame
rewards_category_pct_change = rewards_category.set_index('slot').pct_change().reset_index()

# Merge the DataFrames on the slot column for percent change calculations (elasticity)
elasticity_df = pd.merge(active_validators_category[['slot', 'Solo Stakers']], rewards_category_pct_change, on='slot')

# Merge the DataFrames on the slot column for original values (correlation)
correlation_df = pd.merge(active_validators_category[['slot', 'Solo Stakers']], rewards_category, on='slot')

# Replace infinite values with NaN and drop rows with NaN values for elasticity calculations
elasticity_df.replace([np.inf, -np.inf], np.nan, inplace=True)
elasticity_df.dropna(inplace=True)

# Replace infinite values with NaN for correlation calculations
correlation_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Initialize dictionaries to store the results
elasticity = {}
t_stats = {}
standard_deviations = {}
p_values = {}
counts = {}
correlations = {}

# List of columns to calculate elasticity and correlation for
columns = ['CEX', 'Liquid Restaking', 'Liquid Staking', 'Staking Pools']
reference_column = 'Solo Stakers'

# Calculate elasticity and correlation for each column
for col in columns:
    # Calculate elasticity using percent change values
    elasticity_df[f'elasticity_{col}'] = elasticity_df[reference_column] / elasticity_df[f'{col}']
    
    # Drop NaN values for the specific column for elasticity calculations
    valid_elasticity = elasticity_df[[f'elasticity_{col}', reference_column, f'{col}']].dropna()
    
    # Store the results for elasticity
    elasticity[col] = valid_elasticity[f'elasticity_{col}'].mean()
    standard_deviations[col] = valid_elasticity[f'elasticity_{col}'].std()
    counts[col] = valid_elasticity[f'elasticity_{col}'].count()
    
    # Perform a one-sample t-test against zero
    t_stat, p_value = stats.ttest_1samp(valid_elasticity[f'elasticity_{col}'], 0)
    t_stats[col] = t_stat
    p_values[col] = p_value
    
    # Calculate the correlation coefficient using non-percent change values
    valid_correlation = correlation_df[[reference_column, col]].dropna()
    correlation, _ = stats.pearsonr(valid_correlation[reference_column], valid_correlation[col])
    correlations[col] = correlation

# Print the results
print("Elasticity and Correlation Analysis Results:")
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t-stat = {t_stats[col]}, Std Dev = {standard_deviations[col]}, P-value = {p_values[col]}, Count = {counts[col]}, Correlation Coefficient = {correlations[col]}')

# Display the DataFrame with elasticity columns (optional)
print(elasticity_df)
print(correlation_df)

Elasticity and Correlation Analysis Results:
CEX: Mean Elasticity = -0.000567825168135794, t-stat = -0.10627060671299535, Std Dev = 0.09223794321120013, P-value = 0.9154393944694501, Count = 298, Correlation Coefficient = -0.30869185004218797
Liquid Restaking: Mean Elasticity = -0.0031703156744017046, t-stat = -0.8554572829296357, Std Dev = 0.06397529717419181, P-value = 0.39298762575596, Count = 298, Correlation Coefficient = -0.15036988245762392
Liquid Staking: Mean Elasticity = -0.0057232887976703525, t-stat = -0.8044649589781976, Std Dev = 0.1228136563773882, P-value = 0.4217719684706013, Count = 298, Correlation Coefficient = -0.298597679635162
Staking Pools: Mean Elasticity = -0.013616922121310757, t-stat = -1.0754205413239017, Std Dev = 0.21857916275127842, P-value = 0.2830593366025369, Count = 298, Correlation Coefficient = -0.1935610117063778
          slot  Solo Stakers        CEX  Liquid Restaking  Liquid Staking  \
1    6847200.0      0.000000  -0.922164         -1.522252  